# LGBM Controlled Tuning (one parameter per time)

Tune `LGBMRegressor` in a controlled way: one parameter sweep per experiment, while all other parameters stay fixed.

In [1]:
import sys

sys.path.append("../")

import json
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error, root_mean_squared_log_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split

from src.loader import Loader

In [3]:
SEED = 42
TEST_SIZE = 0.33
CV = 5

In [4]:
loader = Loader()
df = loader.load("../data/processed_data.csv")
df.shape

(4459, 4732)

In [5]:
X = df.drop(columns="target")
y = df["target"]
y_log = np.log1p(y)

(X.shape, y.shape)

((4459, 4731), (4459,))

In [6]:
X_train, X_test, y_train_raw, y_test_raw, y_train_log, y_test_log = train_test_split(
    X,
    y,
    y_log,
    test_size=TEST_SIZE,
    random_state=SEED,
)

cv = KFold(n_splits=CV, shuffle=True, random_state=SEED)

The notebook uses a sequential tuning strategy. Run one sweep, inspect the result, update `current_params` manually only if the change is clearly better, then continue to the next sweep.

In [11]:
current_params = {
    "learning_rate": 0.03,
    "n_estimators": 100,
    "num_leaves": 31,
    "min_child_samples": 20,
    "subsample": 1.0,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.05,
    "reg_lambda": 0.05,
}

current_params

{'learning_rate': 0.03,
 'n_estimators': 100,
 'num_leaves': 31,
 'min_child_samples': 20,
 'subsample': 1.0,
 'colsample_bytree': 0.8,
 'reg_alpha': 0.05,
 'reg_lambda': 0.05}

In [8]:
sweeps = {
    "num_leaves": [7, 15, 31, 63, 127],
    "min_child_samples": [5, 10, 20, 50, 100],
    "learning_rate": [0.03, 0.05, 0.1, 0.15],
    "n_estimators": [100, 300, 700, 1200],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_alpha": [0.0, 0.05, 0.1, 0.5],
    "reg_lambda": [0.0, 0.05, 0.1, 0.5],
}

list(sweeps.keys())

['num_leaves',
 'min_child_samples',
 'learning_rate',
 'n_estimators',
 'subsample',
 'colsample_bytree',
 'reg_alpha',
 'reg_lambda']

In [9]:
def evaluate_params(params: dict) -> tuple[float, float]:
    model = LGBMRegressor(
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1,
        **params,
    )
    # RMSE in log-space is equivalent to RMSLE for the log-target setup.
    scores = -cross_val_score(
        estimator=model,
        X=X_train,
        y=y_train_log,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    )
    return scores.mean(), scores.std()


def run_single_parameter_sweep(parameter_name: str, candidate_values: list, base_params: dict) -> pd.DataFrame:
    sweep_rows = []

    for candidate_value in candidate_values:
        trial_params = base_params.copy()
        trial_params[parameter_name] = candidate_value
        cv_mean, cv_std = evaluate_params(trial_params)
        sweep_rows.append(
            {
                "sweep_parameter": parameter_name,
                "candidate_value": candidate_value,
                "cv_rmsle_mean": cv_mean,
                "cv_rmsle_std": cv_std,
                **trial_params,
            }
        )

    return pd.DataFrame(sweep_rows).sort_values(by="cv_rmsle_mean").reset_index(drop=True)

In [10]:
all_sweep_results = []

for parameter_name, candidate_values in sweeps.items():
    sweep_df = run_single_parameter_sweep(parameter_name, candidate_values, current_params)
    display(sweep_df.style.format({"cv_rmsle_mean": "{:,.4f}", "cv_rmsle_std": "{:,.4f}"}))
    all_sweep_results.append(sweep_df)

current_params

,sweep_parameter,candidate_value,cv_rmsle_mean,cv_rmsle_std,learning_rate,n_estimators,num_leaves,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,num_leaves,15,1.4699,0.0408,0.100000,100,15,20,1.000000,1.000000,0.000000,0.000000
1,num_leaves,31,1.4719,0.0351,0.100000,100,31,20,1.000000,1.000000,0.000000,0.000000
2,num_leaves,63,1.4743,0.0379,0.100000,100,63,20,1.000000,1.000000,0.000000,0.000000
3,num_leaves,127,1.4802,0.0372,0.100000,100,127,20,1.000000,1.000000,0.000000,0.000000
4,num_leaves,7,1.4879,0.0412,0.100000,100,7,20,1.000000,1.000000,0.000000,0.000000


,sweep_parameter,candidate_value,cv_rmsle_mean,cv_rmsle_std,learning_rate,n_estimators,num_leaves,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,min_child_samples,50,1.4672,0.0433,0.100000,100,31,50,1.000000,1.000000,0.000000,0.000000
1,min_child_samples,10,1.4678,0.0381,0.100000,100,31,10,1.000000,1.000000,0.000000,0.000000
2,min_child_samples,20,1.4719,0.0351,0.100000,100,31,20,1.000000,1.000000,0.000000,0.000000
3,min_child_samples,5,1.4749,0.0494,0.100000,100,31,5,1.000000,1.000000,0.000000,0.000000
4,min_child_samples,100,1.4765,0.0558,0.100000,100,31,100,1.000000,1.000000,0.000000,0.000000


,sweep_parameter,candidate_value,cv_rmsle_mean,cv_rmsle_std,learning_rate,n_estimators,num_leaves,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,learning_rate,0.030000,1.4493,0.0384,0.030000,100,31,20,1.000000,1.000000,0.000000,0.000000
1,learning_rate,0.050000,1.4517,0.0388,0.050000,100,31,20,1.000000,1.000000,0.000000,0.000000
2,learning_rate,0.100000,1.4719,0.0351,0.100000,100,31,20,1.000000,1.000000,0.000000,0.000000
3,learning_rate,0.150000,1.4888,0.0333,0.150000,100,31,20,1.000000,1.000000,0.000000,0.000000


,sweep_parameter,candidate_value,cv_rmsle_mean,cv_rmsle_std,learning_rate,n_estimators,num_leaves,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,n_estimators,100,1.4719,0.0351,0.100000,100,31,20,1.000000,1.000000,0.000000,0.000000
1,n_estimators,300,1.5029,0.0308,0.100000,300,31,20,1.000000,1.000000,0.000000,0.000000
2,n_estimators,700,1.5355,0.0299,0.100000,700,31,20,1.000000,1.000000,0.000000,0.000000
3,n_estimators,1200,1.5630,0.0326,0.100000,1200,31,20,1.000000,1.000000,0.000000,0.000000


,sweep_parameter,candidate_value,cv_rmsle_mean,cv_rmsle_std,learning_rate,n_estimators,num_leaves,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,subsample,0.700000,1.4719,0.0351,0.100000,100,31,20,0.700000,1.000000,0.000000,0.000000
1,subsample,0.850000,1.4719,0.0351,0.100000,100,31,20,0.850000,1.000000,0.000000,0.000000
2,subsample,1.000000,1.4719,0.0351,0.100000,100,31,20,1.000000,1.000000,0.000000,0.000000


,sweep_parameter,candidate_value,cv_rmsle_mean,cv_rmsle_std,learning_rate,n_estimators,num_leaves,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,colsample_bytree,0.800000,1.4621,0.0360,0.100000,100,31,20,1.000000,0.800000,0.000000,0.000000
1,colsample_bytree,0.600000,1.4665,0.0395,0.100000,100,31,20,1.000000,0.600000,0.000000,0.000000
2,colsample_bytree,1.000000,1.4719,0.0351,0.100000,100,31,20,1.000000,1.000000,0.000000,0.000000


,sweep_parameter,candidate_value,cv_rmsle_mean,cv_rmsle_std,learning_rate,n_estimators,num_leaves,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,reg_alpha,0.050000,1.4652,0.0380,0.100000,100,31,20,1.000000,1.000000,0.050000,0.000000
1,reg_alpha,0.000000,1.4719,0.0351,0.100000,100,31,20,1.000000,1.000000,0.000000,0.000000
2,reg_alpha,0.500000,1.4733,0.0364,0.100000,100,31,20,1.000000,1.000000,0.500000,0.000000
3,reg_alpha,0.100000,1.4755,0.0294,0.100000,100,31,20,1.000000,1.000000,0.100000,0.000000


,sweep_parameter,candidate_value,cv_rmsle_mean,cv_rmsle_std,learning_rate,n_estimators,num_leaves,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,reg_lambda,0.050000,1.4697,0.0407,0.100000,100,31,20,1.000000,1.000000,0.000000,0.050000
1,reg_lambda,0.500000,1.4718,0.0343,0.100000,100,31,20,1.000000,1.000000,0.000000,0.500000
2,reg_lambda,0.000000,1.4719,0.0351,0.100000,100,31,20,1.000000,1.000000,0.000000,0.000000
3,reg_lambda,0.100000,1.4725,0.0363,0.100000,100,31,20,1.000000,1.000000,0.000000,0.100000


{'learning_rate': 0.1,
 'n_estimators': 100,
 'num_leaves': 31,
 'min_child_samples': 20,
 'subsample': 1.0,
 'colsample_bytree': 1.0,
 'reg_alpha': 0.0,
 'reg_lambda': 0.0}

The loop above does not change `current_params`. After reviewing each sweep table, update `current_params` manually before rerunning later sweeps or before fitting the final model. This keeps the tuning process explicit and auditable.

In [12]:
sweep_results_df = pd.concat(all_sweep_results, ignore_index=True)
sweep_results_df.head()

,sweep_parameter,candidate_value,cv_rmsle_mean,cv_rmsle_std,learning_rate,n_estimators,num_leaves,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,num_leaves,15.0,1.469949,0.040805,0.1,100,15,20,1.0,1.0,0.0,0.0
1,num_leaves,31.0,1.471935,0.035076,0.1,100,31,20,1.0,1.0,0.0,0.0
2,num_leaves,63.0,1.474347,0.037919,0.1,100,63,20,1.0,1.0,0.0,0.0
3,num_leaves,127.0,1.480226,0.037242,0.1,100,127,20,1.0,1.0,0.0,0.0
4,num_leaves,7.0,1.487938,0.041210,0.1,100,7,20,1.0,1.0,0.0,0.0


In [13]:
best_model = LGBMRegressor(
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
    **current_params,
)

best_model.fit(X_train, y_train_log)

y_pred_log = best_model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_pred = np.clip(y_pred, 0, None)

In [14]:
metrics = pd.DataFrame(
    {
        "metric": ["rmsle", "rmse", "mae", "r2"],
        "value": [
            root_mean_squared_log_error(y_test_raw, y_pred),
            root_mean_squared_error(y_test_raw, y_pred),
            mean_absolute_error(y_test_raw, y_pred),
            r2_score(y_test_raw, y_pred),
        ],
    }
)

metrics.style.format({"value": "{:,.4f}"})

,metric,value
0,rmsle,1.4643
1,rmse,"7,513,960.8937"
2,mae,"4,226,236.7092"
3,r2,0.1154


In [15]:
ARTIFACTS_DIR = Path("../artifacts/baseline")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

sweep_results_df.to_csv(ARTIFACTS_DIR / "lgbm_controlled_tuning_results.csv", index=False)

summary = {
    "target_transform": "log1p",
    "primary_metric": "rmsle",
    "tuning_strategy": "one_parameter_at_a_time",
    "best_params": current_params,
    "test_rmsle": float(root_mean_squared_log_error(y_test_raw, y_pred)),
    "test_rmse": float(root_mean_squared_error(y_test_raw, y_pred)),
    "test_mae": float(mean_absolute_error(y_test_raw, y_pred)),
    "test_r2": float(r2_score(y_test_raw, y_pred)),
}

with open(ARTIFACTS_DIR / "lgbm_controlled_tuning_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

## How To Read The Results

- Compare candidates inside one sweep by `cv_rmsle_mean`.
- Update the baseline only when the gain is stable and meaningful, not because of a tiny difference.
- If two values are very close, prefer the simpler or more conservative choice.
- After finishing all single-parameter sweeps, you can run one small local search around the best settings.